# DES and 3DES: Blocks, Rounds, Padding, and Legacy Risk

These examples use the running scenario of St. Isidore Hospital. They are teaching examples: understand the mechanism, then prefer well-reviewed libraries and current protocols in production.

## Goal

DES is obsolete, but it is useful historically because it shows a block cipher as repeated rounds over fixed-size blocks. Real DES uses initial/final permutations, expansion, key mixing, S-box substitution, and permutation across 16 Feistel rounds.

In [ ]:
from Crypto.Cipher import DES, DES3
from Crypto.Util.Padding import pad, unpad
from Crypto.Random import get_random_bytes

record = b"Patient 2048: oncology appointment at 09:30"
print("Plaintext length:", len(record))
print("DES block size:", DES.block_size)

des_key = b"8bytekey"  # DES requires exactly 8 bytes; only 56 bits are effective.
iv = get_random_bytes(DES.block_size)  # CBC needs a fresh unpredictable IV.
cipher = DES.new(des_key, DES.MODE_CBC, iv)  # CBC chains each block to the previous ciphertext block.
ct = cipher.encrypt(pad(record, DES.block_size))  # Padding extends the message to a full block.
print("IV:", iv.hex())
print("Ciphertext:", ct.hex())

decipher = DES.new(des_key, DES.MODE_CBC, iv)  # Decryption needs the same key and IV.
print(unpad(decipher.decrypt(ct), DES.block_size))  # Remove padding after decryption.

## What DES Does Internally

The next cell is not full DES. It is a miniature Feistel network that mirrors the shape of DES: split the block, transform one side with a round key, XOR it into the other side, then swap. The same structure decrypts by applying round keys in reverse.

In [ ]:
def toy_round_function(right, round_key):
    # This is not DES. It is only a keyed mixing function for the toy Feistel demo.
    return ((right ^ round_key) * 0x45D9F3B) & 0xFFFFFFFF

def toy_feistel_encrypt(block64, round_keys):
    # Split one 64-bit block into two 32-bit halves.
    left = (block64 >> 32) & 0xFFFFFFFF
    right = block64 & 0xFFFFFFFF
    trace = []
    for k in round_keys:
        new_left = right  # Feistel swap: old right becomes new left.
        new_right = left ^ toy_round_function(right, k)  # Mix old left with F(old right, key).
        left, right = new_left, new_right
        trace.append((left, right))
    return ((left << 32) | right), trace

def toy_feistel_decrypt(block64, round_keys):
    # Feistel decryption uses the same structure with round keys in reverse order.
    left = (block64 >> 32) & 0xFFFFFFFF
    right = block64 & 0xFFFFFFFF
    for k in reversed(round_keys):
        old_right = left  # Undo the swap from encryption.
        old_left = right ^ toy_round_function(old_right, k)  # Recover the previous left half.
        left, right = old_left, old_right
    return (left << 32) | right

block = int.from_bytes(b"LABRSLT1", "big")  # One 8-byte hospital-flavored block.
keys = [0x11111111, 0x22222222, 0x33333333, 0x44444444]  # Toy round keys.
encrypted, trace = toy_feistel_encrypt(block, keys)
print("Plain block:", hex(block))
for i, (l, r) in enumerate(trace, 1):
    print(f"Round {i}: L={l:08x} R={r:08x}")
print("Encrypted:", hex(encrypted))
print("Decrypted:", toy_feistel_decrypt(encrypted, keys).to_bytes(8, "big"))

## 3DES

3DES applies DES three times. It was a migration path, not a modern choice. New systems should use AES or authenticated encryption modes instead.

In [ ]:
key_3des = DES3.adjust_key_parity(get_random_bytes(24))  # 3DES uses DES keys with parity bits.
iv_3des = get_random_bytes(DES3.block_size)  # Fresh IV for CBC mode.
triple = DES3.new(key_3des, DES3.MODE_CBC, iv_3des)  # 3DES applies DES operations multiple times.
ct3 = triple.encrypt(pad(record, DES3.block_size))
plain3 = unpad(DES3.new(key_3des, DES3.MODE_CBC, iv_3des).decrypt(ct3), DES3.block_size)
print(ct3.hex())
print(plain3)